Week 13 (21/05/2025) - Custom datasets and auto-encoders

Create a custom dataset using Pytorch functionalities.

In [ ]:
# Create a custom dataset for image classification through a dedicated (.csv) file containing a path to the images and the respective classes.

import os
import pandas as pd
from torchvision import read_image
from torch.utils.data import Dataset

# Start by defining the dataset via classes.
class MyDataset(Dataset):
    def __init__(self, labels, img_dir, transform=None): # for creating an instance of the dataset
        self.img_dir = img_dir
        self.transforms = transform
        self.labels = pd.readcsv(labels)
    def __len__(self): # for understanding the length of the dataset
        return len(self.labels)
    def __getitem__(self, index): # for getting an element of the dataset
        imgs_path = os.join(self.img_dir, self.labels.iloc[index, 0]) # create the path of the desired image
        image = read_image(imgs_path) # use the paths to read the image from the disk
        label = self.labels.iloc[index, 1] # read the corresponding label
        # Apply eventual transformations.
        if self.transforms:
            image = self.transforms(image)
        return image, label # return the (transformed) image and the corresponding label

dataset = MyDataset(labels='labels.csv', img_dir='data')

An auto-encoder is a neural network model that consists of an encoder, which compresses the input data into a lower-dimensional representation, and a decoder, which then learns to reconstruct the representation into the original input data. <br>
For this reason, observe that the two layers have mirrored structures as the input and output layers are "swapped" while the hidden layer stays the same in both components.

In [5]:
import torch
from torch.utils.data import Dataloader
from torchvision.transforms import ToTensor
from torch import nn
from torchvision import datasets
import matplotlib.pyplot as plt
import torchmetrics

# Start by importing the MNIST dataset.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor())

device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility

# Then, create the autoencoder class by defining the encoder and decoder structures.
class AutoEnc(nn.Module):
    def __init__(self):
        super.__init__()
        self.encoder = nn.Sequential(
            # nn.Flatten(),
            nn.Linear(28*28, 50)
            nn.ReLU(),
            nn.Linear(50, 25)
            nn.ReLU(),
            nn.Linear(25, 10)
        )
        self.decoder = nn.Sequential(
            nn.Linear(10, 25),
            nn.ReLU(),
            nn.Linear(25, 50),
            nn.ReLU(),
            nn.Linear(50, 28*28),
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# Create an instance of the model and define its hyperparameters.
model = AutoEnc().to(device) # create the model and move it to the GPU
epochs = 3
batch_size = 16
learning_rate = 0.001

# Then, define the loss function (and its optimizer) to use for the training stage.
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), learning_rate)

# Lastly, define the dataloader.
train_dataloader = Dataloader(training_data, batch_size=batch_size)
test_dataloader = Dataloader(test_data, batch_size=batch_size)

# At this point, start by defining the training loop for the training stage.
def training_loop(dataloader, model, loss_fn, optimizer):
    # Start by getting the batch of data to analyze.
    for batch, (X, y) in enumerate(dataloader):
        X = X.view(28, 28)
        X = X.to(device)
        X_reconstructed = model(X) # use the model to derive the reconstructed image
        loss = loss_fn(X_reconstructed, X) # compute the reconstruction error
        loss.backward() # perform the backward pass to upgrade the weights of the model
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            print(f'Current loss: {loss.item()}')

# Similarly, define the testing loop for the testing stage.
def test_loop(dataloader, model):
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            pred = model(X) # get the reconstructed image
            img = pred.view(-1, 28, 28)
            img = img.unsqueeze(1)
            img = img.permute(2, 3, 1, 0).to('cpu') # perform a procedure to print the reconstructed image
            plt.imshow(img[:,:,:,0]) # show the first image
            plt.show()

# Perform the actual training procedure.
for e in range(epochs):
    print(f'Epoch: {e}')
    training_loop(training_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model)

SyntaxError: invalid syntax (3072633886.py, line 22)